## Libraries

In [1]:
# Prevent OpenMP runtime conflict between libraries like PyTorch, TensorFlow, and NumPy on Windows
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchmetrics.classification import MulticlassAccuracy

from tqdm import tqdm
import numpy as np
from prettytable import PrettyTable

from danflow.training import Trainer

## Load Data

In [2]:
data = torch.load('../saved_values/shuttle_data.pt', weights_only=False)

x_train = data["x_train"]
y_train = data["y_train"]

x_valid = data["x_valid"]
y_valid = data["y_valid"]

x_test = data["x_test"]
y_test = data["y_test"]

## Convert to Tensors

In [3]:
x_train = torch.tensor(np.asarray(x_train), dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)

x_valid = torch.tensor(np.asarray(x_valid), dtype=torch.float32)
y_valid = torch.tensor(np.asarray(y_valid), dtype=torch.long)

x_test = torch.tensor(np.asarray(x_test), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

## Data Loader

In [4]:
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=256)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

## MLP Model

The model is a 3-layer MLP with two hidden layers containing 64 and 32 neurons, respectively, using ReLU activation functions. The output layer contains 7 neurons, corresponding to the 7 target classes.

In [5]:
def mlp_model():
    "Initializes multi layer perceptron model"
    in_features = 9
    num_class = 7
    h1 = 64
    h2 = 32
    h3 = 16

    model = nn.Sequential(nn.Linear(in_features, h1),
                           nn.ReLU(),
                           nn.Linear(h1, h2),
                           nn.ReLU(),
                           nn.Linear(h2, h3),
                           nn.ReLU(),
                           nn.Linear(h3, num_class))

    return model


model = mlp_model()
model

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=16, bias=True)
  (5): ReLU()
  (6): Linear(in_features=16, out_features=7, bias=True)
)

## Cross-Entropy Loss
$$
\mathcal{L} = -\log(\hat{y}_{\text{true}})
$$

In [6]:
loss_fn = nn.CrossEntropyLoss()

## Optimizer

In [7]:
optimizer = optim.SGD(model.parameters(),
                      lr=0.01,
                      momentum=0.9,
                      nesterov=True,
                      weight_decay=1e-4)

## Model Verification

### Step 1: Check Forward Path

Calculate loss for one batch

In [8]:
x_batch, y_batch = next(iter(train_loader))

print(f"x_batch shape = {x_batch.shape}")
print(f"y_batch shape = {y_batch.shape}")

outputs = model(x_batch)

print(f"outputs shape = {outputs.shape}")

loss = loss_fn(outputs, y_batch)
print(f"loss = {loss:.4f}")

x_batch shape = torch.Size([128, 9])
y_batch shape = torch.Size([128])
outputs shape = torch.Size([128, 7])
loss = 1.8535


### Step 2: Check Backward Path

Select 5 random batches and overfit the model

In [9]:
mini_train_dataset, _ = random_split(train_dataset, 
                                     (1000, (len(train_dataset)-1000)))

mini_loader = DataLoader(mini_train_dataset, 
                         batch_size=200, 
                         shuffle=True)

In [10]:
accuracy = MulticlassAccuracy(num_classes=7)

trainer = Trainer(model,
        optimizer,
        loss_fn,
        accuracy)

In [11]:
for epoch in range(500):
    with tqdm(total=1, desc=f"Epoch {epoch}", unit="batch") as pbar:
        loss, acc = trainer.train_epoch(mini_loader)

        pbar.set_postfix(
            accuracy=f"{acc:.4f}",
            loss=f"{loss:.4f}"
        )
        pbar.update(1)

Epoch 499: 100%|██████████| 1/1 [00:00<00:00, 20.22batch/s, accuracy=0.7498, loss=0.0076]


## Learning Rate Selection

In [12]:
for lr in [0.1, 0.01, 0.001, 0.0001]:
    print(f"LR={lr}")

    model = mlp_model()

    optimizer = optim.SGD(model.parameters(),
                          lr=lr,
                          weight_decay=1e-4)

    trainer = Trainer(model,
        optimizer,
        loss_fn,
        accuracy)
    for epoch in range(5):
        with tqdm(total=1, desc=f"Epoch {epoch}", unit="batch") as pbar:
                loss, acc = trainer.train_epoch(train_loader)
        
                pbar.set_postfix(
                    accuracy=f"{acc:.4f}",
                    loss=f"{loss:.4f}"
                )
                pbar.update(1)

    print()
    

LR=0.1


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.25s/batch, accuracy=0.4063, loss=0.0966]



LR=0.01


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.25s/batch, accuracy=0.3705, loss=0.2445]



LR=0.001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.54s/batch, accuracy=0.1429, loss=1.2842]



LR=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.25s/batch, accuracy=0.1605, loss=1.8264]

## Small Grid

In [13]:
my_table = PrettyTable(
    ["Learning Rate", "Weight decay", "Accuracy", "Loss"]
)

for lr in [0.1, 0.15, 0.20, 0.25]:
    for wd in [0.0, 1e-4, 1e-5, 1e-6]:

        model = mlp_model()
       
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            weight_decay=wd
        )

        trainer = Trainer(
            model,
            optimizer,
            loss_fn,
            accuracy
        )

        tqdm.write(f"LR={lr}, WD={wd}")

        for epoch in range(5):
            with tqdm(
                total=1,
                desc=f"Epoch {epoch}",
                unit="batch"
            ) as pbar:

                loss, acc = trainer.train_epoch(train_loader)

                pbar.set_postfix(
                    accuracy=f"{acc:.4f}",
                    loss=f"{loss:.4f}"
                )

                pbar.update(1)

        my_table.add_row([
            lr,
            wd,
            f"{100. * acc:.4f}",
            f"{loss:.4f}"
        ])

        tqdm.write("")

    my_table.add_row([
        "-" * 20,
        "-" * 20,
        "-" * 20,
        "-" * 20
    ])

print(my_table)

LR=0.1, WD=0.0


Epoch 0:   0%|          | 0/1 [00:00<?, ?batch/s]

Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.25s/batch, accuracy=0.4266, loss=0.0366]



LR=0.1, WD=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.41s/batch, accuracy=0.4269, loss=0.0425]


...


LR=0.25, WD=1e-05


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.41s/batch, accuracy=0.4259, loss=0.0353]



LR=0.25, WD=1e-06


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.45s/batch, accuracy=0.1429, loss=0.6764]


+----------------------+----------------------+----------------------+----------------------+
|    Learning Rate     |     Weight decay     |       Accuracy       |         Loss         |
+----------------------+----------------------+----------------------+----------------------+
|         0.1          |         0.0          |       42.6567        |        0.0366        |
|         0.1          |        0.0001        |       42.6908        |        0.0425        |
|         0.1          |        1e-05         |       42.6620        |        0.0374        |
|         0.1          |        1e-06         |       42.5934        |        0.0440        |
| -------------------- | -------------------- | -------------------- | -------------------- |
|         0.15         |         0.0          |       42.2663        |        0.0630        |
|         0.15         |        0.0001        |       42.7502        |        0.0670        |
|         0.15         |        1e-05         |       42.34

## Train More Epochs

In [14]:
model = mlp_model()

optimizer = optim.SGD(model.parameters(),
                      lr=0.2,
                      weight_decay=1e-4)

trainer = Trainer(model,
        optimizer=optimizer,
        loss_fn=loss_fn,
        metric=accuracy)

In [15]:
history = trainer.fit(train_loader=train_loader,
            valid_loader=valid_loader,
            epochs=300,
            save_best=True,
            checkpoint_path="/assets")

Epoch:   0%|          | 0/300 [00:00<?, ?epoch/s]

                  Epoch 1                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.9690 │     2.2009 │
│ MulticlassAccuracy │ 0.2595 │     0.1335 │
└────────────────────┴────────┴────────────┘

                  Epoch 2                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 1.0549 │     2.4024 │
│ MulticlassAccuracy │ 0.2233 │     0.1473 │
└────────────────────┴────────┴────────────┘

                  Epoch 3                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.2832 │     2.7348 │
│ MulticlassAccuracy │ 0.2724 │     0.1525 │
└────────────────────┴────────┴────────────┘

                  Epoch 4                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.1890 │     3.1038 │
│ MulticlassAccuracy │ 0.2841 │     0.1504 │
└────────────────────┴────────┴────────────┘

...

                 Epoch 296                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.6697 │     0.6712 │
│ MulticlassAccuracy │ 0.1429 │     0.1429 │
└────────────────────┴────────┴────────────┘

                 Epoch 297                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.6697 │     0.6712 │
│ MulticlassAccuracy │ 0.1429 │     0.1429 │
└────────────────────┴────────┴────────────┘

                 Epoch 298                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.6698 │     0.6712 │
│ MulticlassAccuracy │ 0.1429 │     0.1429 │
└────────────────────┴────────┴────────────┘

                 Epoch 299                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.6697 │     0.6712 │
│ MulticlassAccuracy │ 0.1429 │     0.1429 │
└────────────────────┴────────┴────────────┘

                 Epoch 300                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Measure            ┃  Train ┃ Validation ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Loss               │ 0.6697 │     0.6711 │
│ MulticlassAccuracy │ 0.1429 │     0.1429 │
└────────────────────┴────────┴────────────┘